<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/24_multi_hop_reasoning_rag/multi_hop_reasoning_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Hop Reasoning in RAG

This notebook implements a multi-hop Retrieval-Augmented Generation (RAG) system that answers complex queries by retrieving and combining information from multiple documents.

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Step 1: Prepare Dataset

In [44]:
documents = [
    "Elon Musk is the CEO of Tesla and SpaceX.",
    "SpaceX is a private aerospace company focused on space exploration.",
    "Tesla is an electric vehicle company.",
    "The headquarters of Tesla is in Austin, Texas.",
    "NASA collaborates with private companies for space missions.",
    "Elon Musk founded SpaceX in 2002."
]

df = pd.DataFrame({"text": documents})
df

,text
0,Elon Musk is the CEO of Tesla and SpaceX.
1,SpaceX is a private aerospace company focused ...
2,Tesla is an electric vehicle company.
3,"The headquarters of Tesla is in Austin, Texas."
4,NASA collaborates with private companies for s...
5,Elon Musk founded SpaceX in 2002.


## Step 2: Load Embedding Model

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

## Step 3: Create Embeddings

In [ ]:
doc_embeddings = embedding_model.encode(df['text'].tolist())

## Step 4: Retrieval Function

In [ ]:
def retrieve(query, top_k=2):
    query_embedding = embedding_model.encode([query])
    scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return df.iloc[top_indices], scores[top_indices]

## Step 5: Follow-up Query Generation

In [ ]:
def generate_followup_query(query):
    if "Elon Musk" in query:
        return "Which company of Elon Musk is related to space?"
    return query

## Step 6: Context Cleaning

In [ ]:
def clean_context(context_1, context_2):
    sentences = (context_1 + " " + context_2).split(".")
    unique_sentences = []
    for s in sentences:
        s = s.strip()
        if s and s not in unique_sentences:
            unique_sentences.append(s)
    return ". ".join(unique_sentences)

## Step 7: Answer Extraction

In [ ]:
def extract_answer(context, query):
    context_lower = context.lower()

    if "elon musk" in context_lower and "space" in query.lower():
        if "spacex" in context_lower:
            return "SpaceX"

    return "Answer not found"

## Step 8: Multi-Hop RAG Pipeline

In [ ]:
def multi_hop_rag(query):
    print("🔹 Original Query:", query)

    docs_1, _ = retrieve(query)
    context_1 = " ".join(docs_1['text'].tolist())
    print("\n📄 First Retrieval:\n", context_1)

    followup_query = generate_followup_query(query)
    print("\n🔁 Follow-up Query:\n", followup_query)

    docs_2, _ = retrieve(followup_query)
    context_2 = " ".join(docs_2['text'].tolist())
    print("\n📄 Second Retrieval:\n", context_2)

    final_context = clean_context(context_1, context_2)
    print("\n🧠 Final Combined Context:\n", final_context)

    answer = extract_answer(final_context, query)
    print("\n✅ Final Answer:\n", answer)

    return answer

## Step 9: Test the System

In [45]:
query = "Which company founded by Elon Musk works in space?"
multi_hop_rag(query)

🔹 Original Query: Which company founded by Elon Musk works in space?

📄 First Retrieval:
 Elon Musk founded SpaceX in 2002. Elon Musk is the CEO of Tesla and SpaceX.

🔁 Follow-up Query:
 Which company of Elon Musk is related to space?

📄 Second Retrieval:
 Elon Musk is the CEO of Tesla and SpaceX. Elon Musk founded SpaceX in 2002.

🧠 Final Combined Context:
 Elon Musk founded SpaceX in 2002. Elon Musk is the CEO of Tesla and SpaceX

✅ Final Answer:
 SpaceX


'SpaceX'